# MNIST experiments playground

<div style="padding: 0.75rem 1rem; border-left: 4px solid #22c55e; margin: 1rem 0;">
<strong>Playground.</strong> Interfaz simple para entrenar, forkear, extender y comparar ramas de <code>jarl.experiments</code> con MNIST. TensorBoard se abre embebido en el notebook.
</div>

Inspirado en tu demo TFM (<code>learning/mnist/mnist_dag_training.ipynb</code>). Para un tour completo de la API, usa <code>experiments_mnist_tour.ipynb</code>.


## Prerequisites

- `uv sync --extra examples`
- Ejecutar desde `examples/notebooks/`
- Kernel reiniciado si cambias `notebook_utils.py`


## What you will learn

- Entrenar la cabeza de una rama con sliders
- Crear forks y extends con un clic
- Ver el árbol y métricas al instante
- Abrir TensorBoard en el navegador del notebook


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "examples" / "notebooks").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

EXP_DIR = NOTEBOOK_DIR / "_outputs" / "mnist_playground"
DATA_ROOT = NOTEBOOK_DIR / "_data"
print(f"Experiment dir: {EXP_DIR}")

In [ ]:
from jarl.experiments.run_config import CheckpointConfig, RunConfig, TrackingConfig
from notebook_utils import MNISTPlayground, load_mnist_arrays


class MNISTPlaygroundConfig(RunConfig):
    """Config corta para el playground interaction."""

    DEFAULT_PROJECT_NAME = "jarl-demos"
    DEFAULT_EXP_NAME = "mnist-playground"
    learning_rate: float = 0.02
    hidden_dim: int = 64
    batch_size: int = 128
    train_steps: int = 8
    seed: int = 0


TRAIN_BATCH, TEST_BATCH = load_mnist_arrays(train_size=2048, test_size=512, data_root=str(DATA_ROOT))
BASE_CONFIG = MNISTPlaygroundConfig(
    checkpoint=CheckpointConfig(max_to_keep=8, save_interval_steps=4),
    tracking=TrackingConfig(track_tensorboard=True, track_wandb=False),
)

playground = MNISTPlayground(
    EXP_DIR,
    train_batch=TRAIN_BATCH,
    test_batch=TEST_BATCH,
    config_cls=MNISTPlaygroundConfig,
    base_config=BASE_CONFIG,
)
playground.load()
playground.status()

## Panel interaction

1. Elige rama origen y hyperparámetros
2. **Train head** — entrena la cabeza actual
3. **Fork** — nueva rama desde un checkpoint del padre
4. **Extend** — continúa la misma rama
5. **TensorBoard** — Scalars por rama + árbol del experimento en **Images → experiment_tree**


In [ ]:
import ipywidgets as widgets
from IPython.display import display as show_widget


def _branch_options() -> list[str]:
    return playground.branch_names() or ["main"]


branch_select = widgets.Dropdown(options=_branch_options(), description="Branch")
new_branch = widgets.Text(value="lr_low", description="New branch")
label = widgets.Text(value="run", description="Label")
lr = widgets.FloatSlider(value=0.02, min=0.001, max=0.1, step=0.001, description="LR")
hidden = widgets.Dropdown(options=[64, 128], value=64, description="Hidden")
batch = widgets.Dropdown(options=[64, 128], value=128, description="Batch")
steps = widgets.IntSlider(value=8, min=2, max=24, step=2, description="Steps")
tb_port = widgets.IntText(value=6006, description="TB port")
output = widgets.Output()

train_btn = widgets.Button(description="Train head", button_style="success")
fork_btn = widgets.Button(description="Fork + train", button_style="primary")
extend_btn = widgets.Button(description="Extend + train")
checkout_btn = widgets.Button(description="Checkout")
status_btn = widgets.Button(description="Status")
plot_btn = widgets.Button(description="Plot DAG")
tb_btn = widgets.Button(description="TensorBoard", button_style="info")
reset_btn = widgets.Button(description="Reset experiment", button_style="warning")


def _refresh_branches() -> None:
    names = _branch_options()
    branch_select.options = names
    if names:
        branch_select.value = names[0]


def _kwargs() -> dict[str, float | int]:
    return {
        "learning_rate": lr.value,
        "hidden_dim": hidden.value,
        "batch_size": batch.value,
        "train_steps": steps.value,
    }


def _on_train(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        try:
            playground.train_head(branch_select.value, **_kwargs())
            playground.status()
        except Exception as exc:
            print(type(exc).__name__ + ":", exc)


def _on_fork(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        try:
            playground.fork_branch(
                branch_select.value,
                new_branch.value.strip(),
                label=label.value.strip(),
                **_kwargs(),
            )
            _refresh_branches()
            branch_select.value = new_branch.value.strip()
            playground.status()
        except Exception as exc:
            print(type(exc).__name__ + ":", exc)


def _on_extend(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        try:
            playground.extend_branch(
                branch_select.value,
                label=label.value.strip(),
                **_kwargs(),
            )
            playground.status()
        except Exception as exc:
            print(type(exc).__name__ + ":", exc)


def _on_checkout(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        playground.checkout(branch_select.value)
        playground.status()


def _on_status(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        playground.status()


def _on_plot(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        playground.plot_tree()


def _on_tb(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        selected = [branch_select.value]
        if len(_branch_options()) > 1:
            selected = _branch_options()
        print("logdir_spec:", playground.tensorboard_spec(selected))
        playground.show_tensorboard(selected, port=tb_port.value)


def _on_reset(_btn: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        playground.reset()
        playground.load()
        _refresh_branches()
        playground.status()


for btn, handler in [
    (train_btn, _on_train),
    (fork_btn, _on_fork),
    (extend_btn, _on_extend),
    (checkout_btn, _on_checkout),
    (status_btn, _on_status),
    (plot_btn, _on_plot),
    (tb_btn, _on_tb),
    (reset_btn, _on_reset),
]:
    btn.on_click(handler)

controls = widgets.VBox(
    [
        widgets.HBox([branch_select, new_branch, label]),
        widgets.HBox([lr, hidden, batch, steps]),
        widgets.HBox([train_btn, fork_btn, extend_btn, checkout_btn]),
        widgets.HBox([status_btn, plot_btn, tb_btn, tb_port, reset_btn]),
        output,
    ]
)
show_widget(controls)

## Atajos en código (opcional)

Si prefieres celdas manuals en lugar del panel:

```python
playground.train_head("main", learning_rate=0.02, train_steps=8)
playground.fork_branch("main", "lr_low", learning_rate=0.01, label="lr_0.01")
playground.extend_branch("main", learning_rate=0.03, label="fast")
playground.checkout("lr_low")
playground.plot_tree()
playground.show_tensorboard(["main", "lr_low"])
```


## Takeaways

- El playground **persiste** en `_outputs/mnist_playground` — recarga al reiniciar el kernel
- TensorBoard usa `%tensorboard` embebido; si falla, abre `http://localhost:6006`
- Tras completar un nodo, usa **Extend** (misma rama) o **Fork** (nueva rama)

## References

- `experiments_mnist_tour.ipynb` — tour completo de la API
- TFM: `learning/mnist/mnist_dag_training.ipynb`
